# 01 Data Exploration

This notebook builds an initial exploratory and modeling-ready view of ICU stays by joining core MIMIC-IV tables, checking data quality, and creating simple baseline feature sets.

## Load Source Tables

start with three core tables:
- `patients`: demographics
- `admissions`: hospital-level outcomes and metadata
- `icustays`: ICU-level timing and length of stay

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Load the minimum set of tables needed for an ICU-stay level baseline dataset.
patients = pd.read_csv("../data/mimic-iv-3.1/hosp/patients.csv.gz")
admissions = pd.read_csv("../data/mimic-iv-3.1/hosp/admissions.csv.gz")
icustays = pd.read_csv("../data/mimic-iv-3.1/icu/icustays.csv.gz")

In [ ]:
# Preview each source independently before joining them.
display(patients.head())
display(admissions.head())
display(icustays.head())

## Merge to an ICU-Stay Level Dataset

The target unit of analysis is one row per ICU stay. We merge admissions onto ICU stays using `subject_id` and `hadm_id`, then merge patient demographics using `subject_id`.

In [ ]:
df = (
    icustays.merge(admissions, on=["subject_id", "hadm_id"], validate="many_to_one")
    .merge(patients, on="subject_id", validate="many_to_one")
    .copy()
)

# Basic merge sanity checks to catch accidental row inflation or duplicate ICU stays.
print(f"Merged rows: {len(df):,} | Original ICU rows: {len(icustays):,}")
print("Duplicate stay_id rows:", int(df.duplicated(subset=["stay_id"]).sum()))
display(df[["subject_id", "hadm_id", "stay_id"]].isna().sum().rename("null_count"))

display(df.head())

In [ ]:
df.shape

In [ ]:
df.columns

## Target Definition

The current baseline target is `hospital_expire_flag`, which corresponds to in-hospital mortality rather than ICU-only mortality.

In [ ]:
# Convert the target to an explicit integer column for downstream modeling.
df["mortality"] = df["hospital_expire_flag"].astype(int)

display(df["mortality"].value_counts().rename("count"))
display((df["mortality"].value_counts(normalize=True) * 100).rename("percent"))

In [ ]:
# Quick visualization of the target variable.
df["mortality"].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Mortality")
plt.ylabel("Count")
plt.title("Distribution of Mortality in the Dataset")
plt.show()

In [ ]:
# Age distribution
df["anchor_age"].describe()

In [ ]:
# Gender distribution from the merged patient table
df["gender"].value_counts(normalize=True) * 100

In [ ]:
# ICU length of stay distribution
df["los"].describe()

In [ ]:
# Check key modeling columns for missingness before feature creation.
df[["anchor_age", "gender", "los", "mortality"]].isna().sum()

In [ ]:
df["gender"].value_counts()

## Feature Engineering

Two simple baseline views are created:
- **Retrospective view**: includes full ICU length of stay (`los`)
- **Leakage-safer view**: excludes full `los` for earlier risk estimation scenarios

In [ ]:
# Create numeric baseline features with explicit validation for category mapping.
df["age"] = df["anchor_age"]

# Make the cell robust when run independently.
if "mortality" not in df.columns and "hospital_expire_flag" in df.columns:
    df["mortality"] = df["hospital_expire_flag"].astype(int)

gender_map = {"M": 0, "F": 1}
unknown_gender_mask = ~df["gender"].isin(gender_map)
if unknown_gender_mask.any():
    unknown_vals = sorted(df.loc[unknown_gender_mask, "gender"].dropna().unique().tolist())
    raise ValueError(f"Unexpected gender values found: {unknown_vals}")

df["gender_num"] = df["gender"].map(gender_map)

# Retrospective baseline dataset (includes LOS; may leak if used for early prediction tasks).
df_model_retro = df[["age", "gender_num", "los", "mortality"]].dropna().copy()

# Leakage-safer dataset for earlier risk estimation (excludes full LOS).
df_model_safe = df[["age", "gender_num", "mortality"]].dropna().copy()

In [ ]:
# Retrospective feature set (includes LOS).
features_retro = ["age", "gender_num", "los"]
X_retro = df_model_retro[features_retro]
y_retro = df_model_retro["mortality"]

# Leakage-safer feature set (excludes LOS).
features_safe = ["age", "gender_num"]
X_safe = df_model_safe[features_safe]
y_safe = df_model_safe["mortality"]

In [ ]:
# Preview both feature views and confirm they are null-free after filtering.
display(X_retro.head())
display(df_model_retro[features_retro + ["mortality"]].isna().sum())
print("Retro dataset shape:", df_model_retro.shape)

display(X_safe.head())
display(df_model_safe[features_safe + ["mortality"]].isna().sum())
print("Safe dataset shape:", df_model_safe.shape)

In [ ]:
# Distribution checks with mortality stratification
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_model_retro[df_model_retro["mortality"] == 0]["age"].plot(
    kind="hist", bins=30, alpha=0.6, ax=axes[0], label="Survived"
)
df_model_retro[df_model_retro["mortality"] == 1]["age"].plot(
    kind="hist", bins=30, alpha=0.6, ax=axes[0], label="Died"
)
axes[0].set_title("Age Distribution by Mortality")
axes[0].set_xlabel("Age")
axes[0].legend()

df_model_retro[df_model_retro["mortality"] == 0]["los"].plot(
    kind="hist", bins=30, alpha=0.6, ax=axes[1], label="Survived"
)
df_model_retro[df_model_retro["mortality"] == 1]["los"].plot(
    kind="hist", bins=30, alpha=0.6, ax=axes[1], label="Died"
)
axes[1].set_title("LOS Distribution by Mortality")
axes[1].set_xlabel("Length of Stay (days)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Compact data-quality summary for the retrospective modeling view.
summary_cols = ["age", "gender_num", "los", "mortality"]
dq_summary = pd.DataFrame({
    "dtype": df_model_retro[summary_cols].dtypes.astype(str),
    "missing": df_model_retro[summary_cols].isna().sum(),
    "missing_pct": (df_model_retro[summary_cols].isna().mean() * 100).round(4),
    "unique": df_model_retro[summary_cols].nunique(),
    "min": df_model_retro[summary_cols].min(numeric_only=False),
    "max": df_model_retro[summary_cols].max(numeric_only=False),
})

display(dq_summary)

In [ ]:
# LOS can be right-skewed, so inspect both clipped and log-transformed views.
import numpy as np

los_upper = df_model_retro["los"].quantile(0.99)
los_clipped = df_model_retro["los"].clip(upper=los_upper)
los_log1p = np.log1p(df_model_retro["los"])

print(f"LOS 99th percentile cap: {los_upper:.2f} days")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
los_clipped.plot(kind="hist", bins=40, ax=axes[0], title="LOS (Clipped at 99th pct)")
axes[0].set_xlabel("Length of Stay (days)")

pd.Series(los_log1p, name="log1p_los").plot(
    kind="hist", bins=40, ax=axes[1], title="log1p(LOS) Distribution",
    color="tab:orange"
)
axes[1].set_xlabel("log1p(Length of Stay)")

plt.tight_layout()
plt.show()

## Prediction Timepoint Assumption

This section builds two baseline feature views:
- **Retrospective view**: includes full ICU LOS and is useful for post-hoc benchmarking
- **Leakage-safer view**: excludes full LOS and is better aligned with earlier risk estimation

When moving into modeling, choose the feature set that matches the intended prediction timepoint.

In [ ]:
# Explore whether mortality risk shifts across LOS quartiles.
tmp = df_model_retro[["los", "mortality"]].copy()
tmp["los_quartile"] = pd.qcut(tmp["los"], q=4, duplicates="drop")

quartile_mortality = (
    tmp.groupby("los_quartile", observed=False)["mortality"]
    .agg(n="size", mortality_rate="mean")
    .reset_index()
    .sort_values("los_quartile")
)
quartile_mortality["mortality_rate_pct"] = (quartile_mortality["mortality_rate"] * 100).round(2)

display(quartile_mortality)

ax = quartile_mortality.plot(
    kind="bar",
    x="los_quartile",
    y="mortality_rate_pct",
    legend=False,
    figsize=(8, 4),
    color="tab:red",
    title="Mortality Rate by LOS Quartile"
)
ax.set_xlabel("LOS Quartile")
ax.set_ylabel("Mortality Rate (%)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Validation and Handoff

This final section applies a few lightweight constraints and exports a reproducible handoff dataset for downstream notebooks.

Validation checks cover:
- plausible age range for this cohort
- non-negative ICU length of stay
- binary mortality target
- binary encoded gender feature

In [ ]:
from pathlib import Path
import json

# Strict but simple data validation checks before exporting a handoff dataset.
assert df_model_retro["age"].between(18, 100).all(), "Found age values outside [18, 100]."
assert (df_model_retro["los"] >= 0).all(), "Found negative LOS values."
assert set(df_model_retro["mortality"].unique()).issubset({0, 1}), "Mortality must be binary {0,1}."
assert set(df_model_retro["gender_num"].unique()).issubset({0, 1}), "gender_num must be binary {0,1}."

print("Validation checks passed.")

# Export one of the two baseline views for downstream training notebooks.
handoff_view = "safe"  # choose from: "safe", "retro"

if handoff_view == "safe":
    handoff_df = df_model_safe.copy()
    handoff_features = features_safe.copy()
elif handoff_view == "retro":
    handoff_df = df_model_retro.copy()
    handoff_features = features_retro.copy()
else:
    raise ValueError("handoff_view must be either 'safe' or 'retro'.")

out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / f"01_eda_handoff_{handoff_view}.csv"
meta_path = out_dir / f"01_eda_handoff_{handoff_view}_meta.json"

handoff_df.to_csv(csv_path, index=False)

metadata = {
    "handoff_view": handoff_view,
    "features": handoff_features,
    "target": "mortality",
    "rows": int(handoff_df.shape[0]),
    "columns": handoff_df.columns.tolist(),
}
meta_path.write_text(json.dumps(metadata, indent=2))

print(f"Saved handoff dataset: {csv_path}")
print(f"Saved handoff metadata: {meta_path}")
display(handoff_df.head())